# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MubashshiraKaisar/FlyRank-AI-Tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Enter your Hugging Face Read token: ")
login(token=hf_token)

print("Hugging Face login successful!")

Enter your Hugging Face Read token: ··········
Hugging Face login successful!


In [1]:
import duckdb

con = duckdb.connect()

print("Connection successful!")

Connection successful!


In [5]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN ?
)
""", [hf_token])

print("Dataset access setup complete!")

Dataset access setup complete!


#**1. Unit of analysis + time window**

The raw daily performance table contains one row per reporting date, client, and content item.

For the modeling dataset, one row represents one content item belonging to one client.

Feature period: March 1–31, 2026.

Prediction/outcome period: April 1–30, 2026.

Prediction objective: Identify whether a content item's average daily GSC clicks decrease in April compared with March.

Only content-client pairs with complete GSC records for all 31 March days and all 30 April days are included in the modeling dataset.

In [6]:
daily = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{daily}')
""").df()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [8]:
con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{daily}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [19]:
april = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_items
    FROM read_parquet('{april}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date,unique_clients,unique_content_items
0,10424730,2026-04-01,2026-04-30,61,362172


#**2. Fields: feature / label / context / excluded**

**Objective:**
The goal is to identify existing webpages that may need refreshing.

The prediction is made using information available up to the reporting date. The target is whether a webpage's average daily organic search clicks decrease in April 2026 compared with March 2026. March performance is used as historical information, and April performance determines the outcome label.

Features will contain historical performance information. Labels will represent future performance outcomes. Context fields will identify clients, content items, and dates. Future information and fields that could cause data leakage will be excluded.


**Field classification:**

**Features — information available before the prediction period**
- `march_impressions` — total GSC impressions during March 2026.
- `march_clicks` — total GSC clicks during March 2026.
- `march_avg_position` — average GSC search position during March 2026.
- `march_ctr` — March clicks divided by March impressions.
- `march_recent_clicks` — GSC clicks from March 25–31.

**Label — the outcome to predict**
- `is_declining_label` — 1 if average daily GSC clicks decreased in April compared with March; otherwise 0.

**Context — used to identify or group records**
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content item.
- `report_date` — identifies the reporting date.
- `month` — identifies the reporting month.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — describe data availability.

**Excluded — not used as model features**
- `future_april_clicks` — contains future April performance.
- `future_click_change` — calculated using future April clicks and directly reveals the label.
- `gsc_sum_position` — excluded because average position is used instead.
- GA4 metrics, scroll metrics, and other fields not included in the five-feature model — excluded from this initial model because of availability or missing-data limitations.
- `content_updated_date` and `last_optimized_date` — excluded because many records have dates after March 31, 2026, so they may contain future information.


**Why these five features are valid:**

All five features are calculated using March 2026 GSC performance data, which is available before the April prediction period. They describe historical search performance and do not use future April information.

- `march_impressions`: measures historical search visibility.
- `march_clicks`: measures historical organic search clicks.
- `march_avg_position`: measures historical search ranking position.
- `march_ctr`: measures the proportion of impressions that resulted in clicks.
- `march_recent_clicks`: measures clicks during the final seven days of March.

Only content-client pairs with all 31 days of March GSC data are included.

In [9]:
columns = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{daily}')
""").df()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [15]:
content = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{content}')
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [23]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_sum_position IS NULL
        ) AS missing_gsc_sum_position,
        COUNT(*) FILTER (
            WHERE scroll_events IS NULL
        ) AS missing_scroll_events,
        COUNT(*) FILTER (
            WHERE sessions_organic IS NULL
        ) AS missing_organic_sessions,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet('{daily}')
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_sum_position,missing_scroll_events,missing_organic_sessions,ga4_available_rows
0,3611061,0,1528366,1528366,364347


In [25]:
con.execute(f"""
    SELECT
        COUNT(*) AS content_pairs,
        COUNT(*) FILTER (
            WHERE available_days = 31
        ) AS complete_pairs,
        COUNT(*) FILTER (
            WHERE available_days < 31
        ) AS incomplete_pairs
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS available_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ) t
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_pairs,complete_pairs,incomplete_pairs
0,176738,61796,114942


In [21]:
result = con.execute(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            AVG(gsc_avg_position) AS march_avg_position,
            COUNT(DISTINCT report_date) AS march_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks,
            COUNT(DISTINCT report_date) AS april_days
        FROM read_parquet('{april}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    comparison AS (
        SELECT
            m.*,
            a.april_clicks,
            a.april_days
        FROM march m
        JOIN april_data a
          ON m.client_hash_id = a.client_hash_id
         AND m.content_hash_id = a.content_hash_id
        WHERE m.march_days = 31
          AND a.april_days = 30
    )

    SELECT
        c.client_hash_id,
        c.content_hash_id,
        c.march_impressions,
        c.march_clicks,
        c.march_avg_position,
        d.search_volume,
        d.word_count,
        CASE
            WHEN c.april_clicks / 30.0 < c.march_clicks / 31.0
            THEN 1
            ELSE 0
        END AS is_declining_label,
        c.april_clicks AS future_april_clicks
    FROM comparison c
    JOIN read_parquet('{content}') d
      ON c.client_hash_id = d.client_hash_id
     AND c.content_hash_id = d.content_hash_id
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""").df()

print("Dataset shape:", result.shape)
display(result.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (50310, 9)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,search_volume,word_count,is_declining_label,future_april_clicks
0,client_2094c6eb080311d5,content_d0adf93d84631010,332.0,1.0,7.198547,10,3135,0,1.0
1,client_2094c6eb080311d5,content_d2ace6712a61cc0a,610.0,3.0,17.254789,10,225,1,1.0
2,client_2094c6eb080311d5,content_d301749518bbbf15,1232.0,9.0,6.825186,10,2461,1,1.0
3,client_2094c6eb080311d5,content_d3b2fa9ea6547672,3917.0,5.0,10.749416,0,2685,0,8.0
4,client_2094c6eb080311d5,content_d51e62247b27a1ec,476.0,2.0,8.009238,10,2836,1,0.0


In [22]:
con.register("result_df", result)

con.execute("""
    SELECT
        COUNT(*) AS total_pairs,
        COUNT(*) FILTER (
            WHERE d.content_created_date > DATE '2026-03-31'
        ) AS created_after_march,
        COUNT(*) FILTER (
            WHERE d.content_updated_date > DATE '2026-03-31'
        ) AS updated_after_march,
        COUNT(*) FILTER (
            WHERE d.last_optimized_date > DATE '2026-03-31'
        ) AS optimized_after_march
    FROM result_df r
    JOIN read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    ) d
      ON r.client_hash_id = d.client_hash_id
     AND r.content_hash_id = d.content_hash_id
""").df()

,total_pairs,created_after_march,updated_after_march,optimized_after_march
0,50310,0,39216,24324


In [26]:
model_data = con.execute(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks) AS march_clicks,
            AVG(gsc_avg_position) AS march_avg_position,

            SUM(gsc_clicks) FILTER (
                WHERE report_date >= DATE '2026-03-25'
            ) AS march_recent_clicks,

            COUNT(DISTINCT report_date) AS march_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks,
            COUNT(DISTINCT report_date) AS april_days
        FROM read_parquet('{april}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_impressions,
        m.march_clicks,
        m.march_avg_position,

        CASE
            WHEN m.march_impressions > 0
            THEN m.march_clicks * 1.0 / m.march_impressions
            ELSE NULL
        END AS march_ctr,

        m.march_recent_clicks,

        CASE
            WHEN a.april_clicks / 30.0 < m.march_clicks / 31.0
            THEN 1
            ELSE 0
        END AS is_declining_label,

        a.april_clicks AS future_april_clicks

    FROM march m
    JOIN april_data a
      ON m.client_hash_id = a.client_hash_id
     AND m.content_hash_id = a.content_hash_id

    JOIN read_parquet('{content}') d
      ON m.client_hash_id = d.client_hash_id
     AND m.content_hash_id = d.content_hash_id

    WHERE m.march_days = 31
      AND a.april_days = 30
      AND d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""").df()

print("Dataset shape:", model_data.shape)
display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (50310, 9)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_ctr,march_recent_clicks,is_declining_label,future_april_clicks
0,client_62f4a7e64f5e0096,content_64706c8afebebb8c,666.0,2.0,5.928198,0.003003,0.0,1,1.0
1,client_62f4a7e64f5e0096,content_4b46566297566e02,561.0,0.0,11.943312,0.000000,0.0,0,0.0
2,client_62f4a7e64f5e0096,content_54e631382577b94d,4815.0,6.0,4.943686,0.001246,1.0,1,3.0
3,client_62f4a7e64f5e0096,content_87ccab115e180aa9,884.0,0.0,28.180247,0.000000,0.0,0,0.0
4,client_62f4a7e64f5e0096,content_3a25a0496971e9e7,663.0,2.0,2.451367,0.003017,0.0,1,1.0


In [27]:
model_data[
    [
        "march_impressions",
        "march_clicks",
        "march_avg_position",
        "march_ctr",
        "march_recent_clicks"
    ]
].describe()

,march_impressions,march_clicks,march_avg_position,march_ctr,march_recent_clicks
count,50310.000000,50310.000000,50310.000000,50310.000000,50310.000000
mean,4487.400497,12.774438,12.550495,0.002683,2.841662
std,8471.525658,38.710163,11.616439,0.003518,9.848036
min,60.000000,0.000000,0.376241,0.000000,0.000000
25%,798.000000,1.000000,4.619086,0.000460,0.000000
50%,1915.000000,3.000000,8.094812,0.001724,1.000000
75%,4886.750000,11.000000,17.024018,0.003716,2.000000
max,244931.000000,2506.000000,88.350538,0.135266,585.000000


In [28]:
con.execute(f"""
    WITH recent_days AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS recent_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '2026-03-25'
                              AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS total_pairs,
        COUNT(*) FILTER (
            WHERE recent_days = 7
        ) AS complete_pairs,
        COUNT(*) FILTER (
            WHERE recent_days < 7
        ) AS incomplete_pairs
    FROM recent_days
""").df()

,total_pairs,complete_pairs,incomplete_pairs
0,154416,98069,56347


#**3. Verify it with queries (grain, counts, missing values, windows):**

Every claim above gets a query cell here. A contract claim without a query next to it is a guess.

In [10]:
con.execute(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_gsc_impressions,
        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions,
        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        ) AS missing_ga4_pageviews
    FROM read_parquet('{daily}')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY gsc_data_available, ga4_data_available
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_ga4_sessions,missing_ga4_pageviews
0,False,False,4690323,0,0,0,0
1,False,True,49619,0,0,0,0
2,False,<NA>,1490375,0,0,1490375,1490375
3,True,False,1718348,0,0,0,0
4,True,True,364347,0,0,0,0
5,True,<NA>,1528366,0,0,1528366,1528366


In [11]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS gsc_unavailable_or_unknown_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_unavailable_or_unknown_rows
    FROM read_parquet('{daily}')
""").df()

,total_rows,gsc_available_rows,gsc_unavailable_or_unknown_rows,ga4_available_rows,ga4_unavailable_or_unknown_rows
0,9841378,3611061,6230317,413966,9427412


In [13]:
con.execute(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_gsc_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS missing_gsc_avg_position,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS missing_ga4_sessions,

        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        ) AS missing_ga4_pageviews

    FROM read_parquet('{daily}')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY gsc_data_available, ga4_data_available
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_data_available,ga4_data_available,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_sessions,missing_ga4_pageviews
0,False,False,4690323,0,0,4690323,0,0
1,False,True,49619,0,0,49619,0,0
2,False,<NA>,1490375,0,0,1490375,1490375,1490375
3,True,False,1718348,0,0,0,0,0
4,True,True,364347,0,0,0,0,0
5,True,<NA>,1528366,0,0,0,1528366,1528366


In [14]:
con.execute(f"""
    SELECT
        'gsc_impressions' AS metric,
        gsc_data_available AS availability,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_rows
    FROM read_parquet('{daily}')
    GROUP BY gsc_data_available

    UNION ALL

    SELECT
        'gsc_avg_position',
        gsc_data_available,
        COUNT(*),
        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        )
    FROM read_parquet('{daily}')
    GROUP BY gsc_data_available

    UNION ALL

    SELECT
        'ga4_sessions',
        ga4_data_available,
        COUNT(*),
        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        )
    FROM read_parquet('{daily}')
    GROUP BY ga4_data_available

    UNION ALL

    SELECT
        'ga4_pageviews',
        ga4_data_available,
        COUNT(*),
        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        )
    FROM read_parquet('{daily}')
    GROUP BY ga4_data_available

    ORDER BY metric, availability
""").df()

,metric,availability,total_rows,missing_rows
0,ga4_pageviews,False,6408671,0
1,ga4_pageviews,True,413966,0
2,ga4_pageviews,<NA>,3018741,3018741
3,ga4_sessions,False,6408671,0
4,ga4_sessions,True,413966,0
5,ga4_sessions,<NA>,3018741,3018741
6,gsc_avg_position,False,6230317,6230317
7,gsc_avg_position,True,3611061,0
8,gsc_impressions,False,6230317,0
9,gsc_impressions,True,3611061,0


**Missing data and availability findings:**

The March 2026 partition contains 9,841,378 rows.

- GSC data is available for 3,611,061 rows (36.7%).
- GA4 data is available for 413,966 rows (4.2%).
- Availability flags must be checked using `IS TRUE`.
- GSC impressions are not NULL even when GSC data is unavailable. Therefore, the availability flag must be checked before using these values.
- GSC average position is NULL for all rows where GSC data is unavailable.
- GA4 sessions and pageviews are NULL when GA4 availability is unknown.
- Missing or zero values must not automatically be interpreted as zero performance.
- GSC and GA4 metrics should only be used when their respective availability flags are TRUE.

In [12]:
con.execute(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS total_rows
    FROM read_parquet('{daily}')
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY gsc_data_available, ga4_data_available
""").df()

,gsc_data_available,ga4_data_available,first_date,last_date,total_rows
0,False,False,2026-03-01,2026-03-31,4690323
1,False,True,2026-03-01,2026-03-31,49619
2,False,<NA>,2026-03-01,2026-03-31,1490375
3,True,False,2026-03-01,2026-03-31,1718348
4,True,True,2026-03-01,2026-03-31,364347
5,True,<NA>,2026-03-01,2026-03-31,1528366


**Date-range findings:**

The date-range check found that every GSC/GA4 availability group covers March 1–31, 2026. Availability groups do not have different date ranges within this partition.

In [20]:
con.execute(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks,
            COUNT(DISTINCT report_date) AS march_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS april_clicks,
            COUNT(DISTINCT report_date) AS april_days
        FROM read_parquet('{april}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    comparison AS (
        SELECT
            m.client_hash_id,
            m.content_hash_id,
            m.march_clicks / 31.0 AS march_daily_average,
            a.april_clicks / 30.0 AS april_daily_average
        FROM march m
        JOIN april_data a
          ON m.client_hash_id = a.client_hash_id
         AND m.content_hash_id = a.content_hash_id
        WHERE m.march_days = 31
          AND a.april_days = 30
    )

    SELECT
        COUNT(*) AS total_content_pairs,
        COUNT(*) FILTER (
            WHERE april_daily_average < march_daily_average
        ) AS declining_pairs,
        COUNT(*) FILTER (
            WHERE april_daily_average >= march_daily_average
        ) AS not_declining_pairs
    FROM comparison
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_content_pairs,declining_pairs,not_declining_pairs
0,50311,25779,24532


In [30]:
print("Final modeling dataset:", model_data.shape)

print("\nLabel distribution:")
print(
    model_data["is_declining_label"]
    .value_counts()
    .sort_index()
)

print("\nLabel percentages:")
print(
    model_data["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

Final modeling dataset: (50310, 10)

Label distribution:
is_declining_label
0    24531
1    25779
Name: count, dtype: int64

Label percentages:
is_declining_label
0    48.75969
1    51.24031
Name: proportion, dtype: float64


**Label distribution:**

The final modeling dataset contains 50,310 content-client pairs.

- 25,779 pairs (51.24%) are labeled as declining.
- 24,531 pairs (48.76%) are labeled as not declining.

The two classes are reasonably balanced, so there is no major class imbalance in this dataset.

In [32]:
print("Duplicate content-client pairs:",
      model_data.duplicated(
          subset=["client_hash_id", "content_hash_id"]
      ).sum())

print("\nMissing values in model features and label:")

print(
    model_data[
        [
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "march_ctr",
            "march_recent_clicks",
            "is_declining_label"
        ]
    ].isnull().sum()
)

Duplicate content-client pairs: 0

Missing values in model features and label:
march_impressions      0
march_clicks           0
march_avg_position     0
march_ctr              0
march_recent_clicks    0
is_declining_label     0
dtype: int64


**Final data-quality checks:**

The final modeling dataset passed both integrity checks:

- No duplicate client-content pairs were found.
- No missing values were found in the five model features or the target label.

The final dataset contains 50,310 unique content-client pairs with complete feature and label values.

In [29]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create a feature using future information.
# This is intentionally included only to demonstrate leakage.
model_data["future_click_change"] = (
    model_data["future_april_clicks"] / 30
    - model_data["march_clicks"] / 31
)

features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ctr",
    "march_recent_clicks"
]

X = model_data[features]
y = model_data["is_declining_label"]
groups = model_data["client_hash_id"]

# Use the same client-based split for both models.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

# Model A: valid historical features
model_a = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_a.fit(X.iloc[train_idx], y.iloc[train_idx])

pred_a = model_a.predict(X.iloc[test_idx])

score_a = accuracy_score(
    y.iloc[test_idx],
    pred_a
)

# Model B: intentionally leaked future information
X_leak = model_data[["future_click_change"]]

model_b = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

model_b.fit(
    X_leak.iloc[train_idx],
    y.iloc[train_idx]
)

pred_b = model_b.predict(
    X_leak.iloc[test_idx]
)

score_b = accuracy_score(
    y.iloc[test_idx],
    pred_b
)

print("Model A — Valid features:", round(score_a, 4))
print("Model B — Leaked feature:", round(score_b, 4))
print("Score improvement:", round(score_b - score_a, 4))

Model A — Valid features: 0.7348
Model B — Leaked feature: 1.0
Score improvement: 0.2652


**Data leakage experiment:**

Two Decision Tree models were compared using the same client-based train-test split.

- **Model A:** Used five historical March features and achieved 73.48% accuracy.
- **Model B:** Used `future_click_change`, a feature calculated using April clicks, and achieved 100% accuracy.

The leaked feature improved accuracy by 26.52 percentage points. However, this result is misleading because April information is only available after the prediction period.

Therefore, `future_click_change` and `future_april_clicks` are excluded from the actual model. Only historical information available at prediction time should be used.

#**4. Data limits**

**Incomplete historical data:**
Only 61,796 out of 176,738 content-client pairs with GSC data had complete records for all 31 days of March. This means many content items have incomplete historical data. The analysis uses only pairs with complete March history, which reduces the number of available records and may exclude newer or less consistently tracked content.

**Content eligibility:**
One eligible content-client pair was excluded from the final modeling dataset because its metadata showed is_published = FALSE. The content was not deleted and had a metadata record, but it did not meet the published-content eligibility condition.

The final dataset therefore contains 50,310 pairs instead of the 50,311 pairs identified in the initial March–April comparison.

**Metadata timing limitation:**
The content metadata snapshot contains records with update and optimization dates after March 31, 2026. Therefore, these fields cannot automatically be assumed to represent information available at the prediction time.

They are excluded from the initial model to avoid potential data leakage.

In [16]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids,
        MIN(content_created_date) AS earliest_created_date,
        MAX(content_created_date) AS latest_created_date,
        MIN(content_updated_date) AS earliest_updated_date,
        MAX(content_updated_date) AS latest_updated_date
    FROM read_parquet('{content}')
""").df()

,total_rows,unique_content_ids,earliest_created_date,latest_created_date,earliest_updated_date,latest_updated_date
0,519606,519606,2024-10-16,2026-07-06,2024-10-28,2026-07-06


In [17]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE d.report_date < c.content_created_date
        ) AS rows_before_content_creation,
        COUNT(*) FILTER (
            WHERE c.is_deleted IS TRUE
        ) AS deleted_content_rows,
        COUNT(*) FILTER (
            WHERE c.is_published IS TRUE
        ) AS published_content_rows
    FROM read_parquet('{daily}') d
    JOIN read_parquet('{content}') c
      ON d.client_hash_id = c.client_hash_id
     AND d.content_hash_id = c.content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_before_content_creation,deleted_content_rows,published_content_rows
0,9841378,76562,201090,9532718


In [18]:
con.execute(f"""
    SELECT
        is_published,
        is_deleted,
        COUNT(*) AS content_count
    FROM read_parquet('{content}')
    GROUP BY is_published, is_deleted
    ORDER BY is_published, is_deleted
""").df()

,is_published,is_deleted,content_count
0,False,False,6507
1,False,True,101559
2,True,False,411540


In [31]:
con.register("model_data_df", model_data)

con.execute(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS march_days
        FROM read_parquet('{daily}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    april_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(DISTINCT report_date) AS april_days
        FROM read_parquet('{april}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),

    eligible AS (
        SELECT
            m.client_hash_id,
            m.content_hash_id
        FROM march m
        JOIN april_data a
          ON m.client_hash_id = a.client_hash_id
         AND m.content_hash_id = a.content_hash_id
        WHERE m.march_days = 31
          AND a.april_days = 30
    )

    SELECT
        e.client_hash_id,
        e.content_hash_id,
        d.is_published,
        d.is_deleted,
        CASE
            WHEN d.content_hash_id IS NULL THEN 'No metadata record'
            ELSE 'Metadata record exists'
        END AS metadata_status
    FROM eligible e
    LEFT JOIN read_parquet('{content}') d
      ON e.client_hash_id = d.client_hash_id
     AND e.content_hash_id = d.content_hash_id
    LEFT JOIN model_data_df md
      ON e.client_hash_id = md.client_hash_id
     AND e.content_hash_id = md.content_hash_id
    WHERE md.content_hash_id IS NULL
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,is_published,is_deleted,metadata_status
0,client_157ffe4d4a595515,content_19daa2f24df1882d,False,False,Metadata record exists


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.